# Lab 32 · Use gateway models in Microsoft Foundry projects

Connect an app team's Microsoft Foundry project to your Azure AI Gateway, so that the project's agents and code call
the gateway's `gpt-5-mini` alias with the project's managed identity. Then cap the project's tokens with a gateway
policy and read the `429` that Foundry returns. This notebook is the Python track of the [lab guide](README.md): it
deploys the same [infra/foundry-project.bicep](infra/foundry-project.bicep) as the Bicep + azd track and calls the
project like [infra/call_model.py](infra/call_model.py).

The project has no model deployments and holds no key. Foundry shows the alias as the connected model
`<connection>/<alias>`, and calls the gateway with a token of the project's identity.

**Check the alias → Deploy the project and connection → Check the role → Call the model → Run an agent → Try Chat
Completions → Add a token limit → Burst → Inspect the 429 → Clean up**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| Alias `gpt-5-mini` | The alias from notebook 02. The project calls it as `<gateway>/gpt-5-mini`. |
| Resource group `rg-aigw-app-team` | Holds the app team's resources, apart from the gateway's. |
| Foundry resource `<gateway>-apps` and project `app-team` | The app team's Foundry project, with a system-assigned managed identity and no model deployments. |
| Connection `<gateway>` | The gateway connection: category `ApiManagement`, target `<gateway URL>/default/models/openai/v1`, and authentication `ProjectManagedIdentity`. |
| AI Gateway Models User | The project's identity holds it on the gateway's default workspace, so the gateway accepts the project's calls. |
| Foundry User | You hold it on the app team's Foundry resource, so that you can call the project from code. |
| [infra/foundry-project.bicep](infra/foundry-project.bicep) | Declares the Foundry resource, the project, the connection, and both roles. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, Bicep deployments, and retries. |

![The app team's Foundry project has a gateway connection and no model deployments. Foundry calls the gateway with the project's managed identity, and the gateway routes the alias to the backend model](../assets/diagrams/foundry-connection.svg)

## Prerequisites

- An Azure subscription where you can create resource groups and Foundry resources, and assign roles on them and on
  the gateway, for example **Owner**, or **Contributor** with **User Access Administrator**.
- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`. The notebook
  deploys Bicep with `az deployment group create`, and the CLI installs Bicep the first time it needs it.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [01-provision-gateway.ipynb](../01-provision-gateway/01-provision-gateway.ipynb) and
  [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) have run, so the gateway publishes
  the `gpt-5-mini` alias.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it updates the same resources.

In [ ]:
%pip install -q -r infra/requirements.txt requests

### 0. Initialize the lab

The defaults match notebooks 01 and 02, so `utils.Lab` finds the gateway without edits. The app team's resources go
to their own resource group. The Foundry resource's name is also its subdomain, so it must be globally unique: change
`app_account` if the deployment reports that the name is taken.

In [ ]:
import sys
import time
from IPython.display import display

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("32-foundry-gateway-connection")
alias       = "gpt-5-mini"                  # alias from notebook 02 that the project calls
app_rg      = "rg-aigw-app-team"            # the app team's resource group
app_account = f"{lab.gateway_name}-apps"    # the app team's Foundry resource, also its globally unique subdomain
project     = "app-team"                    # the app team's Foundry project
FOUNDRY_API = "2025-06-01"                  # Foundry projects and connections

### 1. Check the alias

The cell confirms that notebook 02 published the alias, and reads the backend model behind it. The connection lists
the model's name and version, and the token limit in step 7 goes on the backend model.

In [ ]:
alias_resource = lab.require(f"{lab.workspace_id}/aliases/{alias}", "Run 02-first-model-call.ipynb first.")
model_id   = alias_resource["properties"]["backendModels"][0]["resourceId"]
model_name = model_id.split("/modelProviders/")[1].replace("/models/", "/")   # <provider>/<model>
deployment = lab.arm("GET", model_id)["properties"]["deployment"]

display(brand.info(f"Alias {alias}", [
    ("Backend model", brand.mono(model_name)),
    ("Model and version", brand.mono(f"{deployment['modelName']} {deployment.get('modelVersion') or ''}".strip())),
    ("Gateway endpoint", brand.mono(f"{lab.gateway_url}/default/models/openai/v1")),
]))

### 2. Deploy the app team's project and connection

[infra/foundry-project.bicep](infra/foundry-project.bicep) creates the app team's side of the hub and spoke, the same
resources that the portal and Azure CLI tracks create:

1. The Foundry resource, with projects turned on and local authentication off.
2. The project `app-team`, with a system-assigned managed identity.
3. The connection, named after the gateway like in the Foundry portal. Its metadata lists the alias with its model
   name and version, and `upstreamSurface` `responses` sends the project's requests to the gateway's Responses API.
4. **AI Gateway Models User** for the project's identity on all of the gateway's assets, with lab 11's module.
5. **Foundry User** for you on the Foundry resource, so that you can call the project from code.

The deployment takes two to three minutes.

In [ ]:
lab.create_resource_group(app_rg)
principal_id, principal_type, _ = lab.signed_in_identity()
outputs = lab.deploy("infra/foundry-project.bicep", resource_group=app_rg,
                     gatewayName=lab.gateway_name, gatewayResourceGroup=lab.resource_group, aliasName=alias,
                     modelName=deployment["modelName"], modelVersion=deployment.get("modelVersion") or "2025-08-07",
                     accountName=app_account, projectName=project, tags=lab.tags,
                     principalId=principal_id, principalType=principal_type)
project_endpoint = outputs["projectEndpoint"]
foundry_model    = outputs["foundryModel"]   # <connection>/<alias>: the name of the connected model in Foundry

account_id = (f"/subscriptions/{lab.subscription_id}/resourceGroups/{app_rg}"
              f"/providers/Microsoft.CognitiveServices/accounts/{app_account}")
connection = lab.arm("GET", f"{account_id}/projects/{project}/connections/{outputs['connectionName']}",
                     api_version=FOUNDRY_API)["properties"]
display(brand.info(f"Connection {outputs['connectionName']}", [
    ("Category", brand.mono(connection["category"])),
    ("Target", brand.mono(connection["target"])),
    ("Authentication", brand.mono(connection["authType"])),
    ("Upstream API", brand.mono(connection["metadata"].get("upstreamSurface", "chat completions"))),
    ("Model in Foundry", brand.mono(foundry_model)),
    ("Project endpoint", brand.mono(project_endpoint)),
], sub=f"{app_account}/{project}"))

### 3. Check the project's role on the gateway

The project's managed identity, not you, calls the gateway. The cell lists the AI Gateway roles that the identity
holds on the gateway.

In [ ]:
project_principal = outputs["projectPrincipalId"]
roles = lab.role_assignments(project_principal)
display(brand.table("The project's roles on the gateway", ["Role", "Scope"],
                    [(r["role"], brand.mono(r["scope"].split("/aigateways/")[1])) for r in roles],
                    sub=f"Managed identity {project_principal}"))
assert any(r["role"] == "AI Gateway Models User" for r in roles), "The project's identity has no role on the gateway."

### 4. Call the model through the project

`AIProjectClient.get_openai_client()` returns an OpenAI client for the project, signed in with your `az login`. The
model is `<connection>/<alias>`: Foundry finds the connection, gets a token for the project's identity, and calls the
gateway's Responses API. New role assignments take a few minutes to apply, so the cell retries `401`, `403`, and
`404` for up to 10 minutes.

In [ ]:
from azure.ai.projects import AIProjectClient

project_client = AIProjectClient(endpoint=project_endpoint, credential=lab.credential)
openai_client  = project_client.get_openai_client()

QUESTION = "In one sentence, what does an AI gateway do for an app team?"
response = utils.retry(lambda: openai_client.responses.create(model=foundry_model, input=QUESTION),
                       statuses=(401, 403, 404), minutes=10, wait=30)
display(brand.reply(response.output_text, "The gateway's model, through the project", model=response.model,
                    tokens=(response.usage.input_tokens, response.usage.output_tokens), sub=QUESTION))

### 5. Run a Foundry agent on the gateway model

A prompt agent in Foundry Agent Service uses the connected model like one of the project's deployments. The cell
creates the agent `gateway-helper-nb`, asks it a question with an `agent_reference`, and deletes it.

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent = project_client.agents.create_version(
    agent_name="gateway-helper-nb",
    definition=PromptAgentDefinition(model=foundry_model, instructions="You answer in one short sentence."))
try:
    answer = openai_client.responses.create(
        input="Why route an agent's model calls through a gateway?",
        extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}})
    display(brand.reply(answer.output_text, f"Agent {agent.name}", model=foundry_model,
                        tokens=(answer.usage.input_tokens, answer.usage.output_tokens), sub=f"Version {agent.version}"))
finally:
    project_client.agents.delete_version(agent_name=agent.name, agent_version=agent.version)
    display(brand.note(f"Deleted agent {agent.name} version {agent.version}", "ok"))

### 6. Try Chat Completions

Foundry serves connected models through the Responses API. The project's Chat Completions endpoint doesn't find them
and answers `404` with `DeploymentNotFound`. The message is Foundry's generic one: the model is a connection, not a
deployment, so waiting doesn't help. Call connected models with the Responses API, as in steps 4 and 5.

In [ ]:
import requests

chat = requests.post(f"{project_endpoint}/openai/v1/chat/completions", timeout=60,
                     headers={"Authorization": f"Bearer {lab.token('https://ai.azure.com/.default')}"},
                     json={"model": foundry_model, "messages": [{"role": "user", "content": "Hi"}]})
display(brand.http(chat.status_code, chat.headers, chat.text, "Chat Completions through the project",
                   sub=foundry_model, keys=["content-length", "x-ms-region"]))
assert chat.status_code == 404 and chat.json()["error"]["code"] == "DeploymentNotFound"

### 7. Add a token limit

The platform team governs the project's calls at the gateway, without touching the project. The cell replaces the
backend model's token limit with **300 tokens per minute for each caller identity**, and keeps its other policies.
It saves the model's earlier token limit, such as notebook 02's 10,000 tokens per minute, so that the clean-up step
restores it. Each project calls with its own managed identity, so each project gets its own 300 tokens.

The limit reaches the runtime in up to a minute, so the cell then waits a minute.

In [ ]:
def set_token_limits(limits):
    """Replace the backend model's token limits, keep its other policies, and return the policies."""
    current = lab.arm("GET", model_id)["properties"]
    others = [p for p in current.get("policies") or [] if p["type"] != "tokenLimit"]
    keep = {k: current[k] for k in ("displayName", "description", "supportedEndpoints", "deployment") if k in current}
    lab.arm("PUT", model_id, {"properties": {**keep, "policies": others + limits}})
    return lab.arm("GET", model_id)["properties"].get("policies") or []

if "earlier_limits" not in globals():   # a rerun keeps the first run's record
    earlier_limits = [p for p in lab.arm("GET", model_id)["properties"].get("policies") or []
                      if p["type"] == "tokenLimit"]
display(brand.policies(model_name, set_token_limits(
    [{"type": "tokenLimit", "period": "minute", "count": 300, "counterKey": ["identity"]}])))
brand.wait(60, "for the limit to reach the runtime")

### 8. Reach the limit through the project

Send four requests in a row through the project. Each one uses about 300 tokens. The OpenAI client retries a `429`
twice by default, which would hide it, so the burst turns that off with `with_options(max_retries=0)`. The gateway
counts a request's tokens after it answers, so one or two requests pass, then the gateway answers `429`, and Foundry
passes it on.

In [ ]:
from openai import RateLimitError

burst_client = openai_client.with_options(max_retries=0)
for attempt in range(1, 4):
    board = brand.Board(f"Burst through project {project}", ["Request", "Status", "Retry-After", "Tokens used"],
                        sub=f"{foundry_model} · attempt {attempt}", budget=300,
                        budget_label="Tokens used / allowance per minute")
    throttled = None
    for number in range(1, 5):
        try:
            result = burst_client.responses.create(model=foundry_model, input=QUESTION)
            board.add([number, brand.status(200), "-", result.usage.total_tokens], 200, result.usage.total_tokens)
        except RateLimitError as error:
            throttled = error
            board.add([number, brand.status(429), error.response.headers.get("retry-after", "-"), "-"], 429)
    if throttled:
        break
    display(brand.note("No 429 yet. The limit may still be applying.", "warn"))
    brand.wait(30, "before the next attempt")

assert throttled, "Expected a 429. Wait a minute and rerun the cell."

### 9. Inspect the throttled response

Foundry returns the gateway's `429` to the caller. The body carries Foundry's `rate_limit_exceeded` code, with the
gateway's message `Token limit is exceeded. Try again in N seconds.` inside it, and `Retry-After` tells the client how
many seconds to wait. The gateway rejected the request itself: the backend model didn't receive it.

In [ ]:
display(brand.http(throttled.status_code, throttled.response.headers, throttled.response.text,
                   "A throttled response through the project", sub=foundry_model, keys=["retry-after", "content-type"]))

### 10. Clean up resources

The cell restores the backend model's earlier token limit, and removes the project's role on the gateway while the
project's identity still exists. Then it deletes the project with its connection, deletes the app team's Foundry
resource, purges it to free its name, and deletes the app team's resource group. The gateway and the alias stay: the
next labs reuse them.

In [ ]:
display(brand.policies(model_name, set_token_limits(earlier_limits)))
lab.delete(outputs["roleAssignmentId"], api_version=utils.ROLES_API)

lab.delete(f"{account_id}/projects/{project}", api_version=FOUNDRY_API, wait=True)
lab.delete(account_id, api_version=FOUNDRY_API)
deleted = (f"/subscriptions/{lab.subscription_id}/providers/Microsoft.CognitiveServices/locations/{lab.location}"
           f"/resourceGroups/{app_rg}/deletedAccounts/{app_account}")
for _ in range(30):   # the soft-deleted account takes a few seconds to appear
    if lab.arm("GET", deleted, api_version=FOUNDRY_API):
        lab.arm("DELETE", deleted, api_version=FOUNDRY_API)
        display(brand.note(f"Purged {app_account}", "ok"))
        break
    time.sleep(10)
lab.delete(f"/subscriptions/{lab.subscription_id}/resourceGroups/{app_rg}", api_version=utils.RESOURCES_API, wait=True)

## Troubleshooting

- **`403` with `Access to model is denied`** in step 4: the project's role on the gateway hasn't applied yet. The cell
  retries for 10 minutes. If it still fails, check step 3: the project's identity needs **AI Gateway Models User**.
- **`404` with `DeploymentNotFound`** on the Responses API: the model isn't `<connection>/<alias>`, or the alias isn't
  in the connection's model list. Print `foundry_model` and compare it with the alias.
- **The deployment fails because the name is taken**: another Foundry resource uses `app_account` as its subdomain,
  or a soft-deleted one holds it. Change `app_account` in step 0, or purge the deleted resource.
- **No `429` in step 8**: the limit takes up to a minute to apply. Wait a minute and rerun the cell.
- **The burst waits instead of failing**: the OpenAI client honors `Retry-After` and retries. Keep
  `with_options(max_retries=0)`.
- **`CannotDeleteResource` when you delete the Foundry resource**: delete its projects first, as the clean-up cell does.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-32-foundry-gateway-connection).

## Next steps

- Run your own agent code in Foundry with the gateway's models and MCP tools in
  [Deploy a Foundry hosted agent behind AI Gateway](../16-foundry-hosted-agent/README.md).
- Set limits per hour or day and per IP address in
  [Throttle tokens per caller with token limits](../06-token-limits/README.md).
- Scope a project's role to one alias in [Go keyless with Microsoft Entra ID and RBAC](../11-entra-rbac/README.md).
- Docs: [Connect Microsoft Foundry to AI Gateway](https://ai.gateway.azure.com/docs/foundry) and
  [Connect Microsoft Foundry agents to AI Gateway](https://learn.microsoft.com/azure/foundry/agents/how-to/ai-gateway).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/32-foundry-gateway-connection/](https://azure.github.io/aigateway/labs/32-foundry-gateway-connection/).